In [ ]:
import cv2
import numpy as np
import os

# ============================================
# CONFIGURAÇÕES
# ============================================

# Lista de pessoas
pessoas = ['caio', 'edson', 'nicolas']

# Variações: sem_avatar e com_avatar
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens
extensao = '.jpg'

# Tipos de ruído (para as imagens ruidosas)
# As imagens originais são apenas o nome base
# As ruidosas terão sufixos: _gauss e _sp
ruidos = ['gauss', 'sp']

# Parâmetro c da equação (Gonzalez recomenda c = -1 para a máscara padrão)
c = -1

# Tamanho do kernel para o Laplaciano (usar 3, pois é o padrão)
laplacian_ksize = 3

# ============================================
# PROCESSAMENTO
# ============================================

print("Iniciando aumento de nitidez com Laplaciano...")

for pessoa in pessoas:
    for versao in versoes:
        # Nome base (original)
        nome_base = f"{pessoa}_{versao}{extensao}"
        
        # Verifica se a original existe
        if not os.path.exists(nome_base):
            print(f"Original não encontrada: {nome_base}. Pulando...")
            continue

        # Carrega imagem original
        img_original = cv2.imread(nome_base)
        if img_original is None:
            print(f"Erro ao ler {nome_base}")
            continue

        # Converte para escala de cinza (o Laplaciano é aplicado em cinza)
        gray_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

        # Calcula Laplaciano da original
        lap_original = cv2.Laplacian(gray_original, cv2.CV_64F, ksize=laplacian_ksize)

        # Aplica equação: g = f + c * Laplaciano
        # Como lap pode ter valores negativos, fazemos a conversão cuidadosamente
        sharp_original = gray_original + c * lap_original
        # Clipa para 0-255 e converte para uint8
        sharp_original = np.clip(sharp_original, 0, 255).astype(np.uint8)

        # Salva imagem nítida da original
        nome_saida = f"sharp_{pessoa}_{versao}.jpg"
        cv2.imwrite(nome_saida, sharp_original)
        print(f"Salvo: {nome_saida}")

        # Agora processa as imagens ruidosas (gauss e sp)
        for ruido in ruidos:
            nome_ruido = f"{pessoa}_{versao}_{ruido}{extensao}"
            if not os.path.exists(nome_ruido):
                print(f"Ruído não encontrado: {nome_ruido}. Pulando...")
                continue

            img_ruido = cv2.imread(nome_ruido)
            if img_ruido is None:
                print(f"Erro ao ler {nome_ruido}")
                continue

            gray_ruido = cv2.cvtColor(img_ruido, cv2.COLOR_BGR2GRAY)
            lap_ruido = cv2.Laplacian(gray_ruido, cv2.CV_64F, ksize=laplacian_ksize)
            sharp_ruido = gray_ruido + c * lap_ruido
            sharp_ruido = np.clip(sharp_ruido, 0, 255).astype(np.uint8)

            nome_saida = f"sharp_{pessoa}_{versao}_{ruido}.jpg"
            cv2.imwrite(nome_saida, sharp_ruido)
            print(f"Salvo: {nome_saida}")

print("Processamento concluído!")